In [4]:
###
#1.
###

from pathlib import Path
import re
import pandas as pd

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Subset


DATASET_ROOT = Path("../data_resized/")

pat = re.compile(
    r"^(?P<actor>[^_]+)_(?P<action>.+)_HD_instance(?P<instance>\d+)_(?P<frame>\d+)\.png$",
    re.I,
)

rows = []
for p in sorted(DATASET_ROOT.rglob("*.png")):
    if "__MACOSX" in p.parts or p.name.startswith("._"):
        continue  # ignore archive artefacts
    m = pat.match(p.name)
    if not m:
        continue
    rows.append(
        dict(
            path=str(p),
            gesture=p.relative_to(DATASET_ROOT).parts[
                0
            ],  # label = top-level folder
            actor=m["actor"],
            instance=int(m["instance"]),
            frame=int(m["frame"]),
            instance_uid=f'{m["actor"]}|{p.relative_to(DATASET_ROOT).parts[0]}|{m["instance"]}',
        )
    )

df = pd.DataFrame(rows)
print(f"frames    : {len(df)}")
print(f"instances : {df['instance_uid'].nunique()}")
print(f"actors    : {df['actor'].nunique()}")
print(f"classes   : {df['gesture'].nunique()}")

# integrity check: every instance should have exactly 5 frames
bad = df.groupby("instance_uid")["frame"].nunique()
assert (
    bad == 5
).all(), f"instances without exactly 5 frames: {bad[bad != 5].index.tolist()}"
print("OK: every instance has 5 frames")

ModuleNotFoundError: No module named 'matplotlib'

In [17]:
###
#2.
###

#####################################
#set up train test split
#####################################



from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


#set up the data, transform to a tensor and organise training and test data
#20% is test and 80% are training
#Extract class names and create a mapping to integers
CLASS_NAMES = sorted(df['gesture'].unique().tolist())
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASS_NAMES)}
#Split by 'instance_uid' frames from the same sequence
unique_instances = df['instance_uid'].unique()
train_instances, test_instances = train_test_split(
    unique_instances, test_size=0.2, random_state=42
)

train_df = df[df['instance_uid'].isin(train_instances)].reset_index(drop=True)
test_df = df[df['instance_uid'].isin(test_instances)].reset_index(drop=True)

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
###
#3.
###

#Training Utilities
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        outputs = model(x)
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct, total = 0, 0

    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item() * y.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


def run_experiment(model, optimizer, criterion, train_loader, val_loader,
                   epochs=10, name='Model'):
    """Run a training experiment and return history dict."""
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} — '
                  f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
                  f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

    return history

In [ ]:
###
#4.
###

#pytorch gestures dataset
class GestureSequenceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        #group by instance so each index grabs 1 sequence of images
        self.instances = list(dataframe.groupby('instance_uid'))
        self.transform = transform
        
    def __len__(self):
        #Returns the total number of frames
        return len(self.instances)
    
    def __getitem__(self, idx):
        #get the row data
        uid, instance_df = self.instances[idx]

        instance_df = instance_df.sort_values('frame')

        frames = []
        for _, row in instance_df.iterrows():
            img = Image.open(row['path']).convert("RGB")
            if self.transform:
                img = self.transform(img)
            frames.append(img)
            
        #stack the five frames into a single tensor shape (5, C, H, W)
        sequence_tensor = torch.stack(frames)

        #frames in the sequence have the same label
        label_str = instance_df.iloc[0]['gesture']
        label = class_to_idx[label_str]

        return sequence_tensor, label

#plot experiments
def plot_experiments(results, show_train=True, show_val=True):
    """Plot training and validation loss + accuracy for multiple experiments."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    ax = axes[0]
    for name, hist in results.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        if show_train:
            ax.plot(epochs, hist['train_loss'], label=f'{name} (Train)', marker='o', markersize=4)
        if show_val:
            ax.plot(epochs, hist['val_loss'], label=f'{name} (Val)', marker='x', markersize=4, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

    # Accuracy
    ax = axes[1]
    for name, hist in results.items():
        epochs = range(1, len(hist['train_acc']) + 1)
        if show_train:
            ax.plot(epochs, hist['train_acc'], label=f'{name} (Train)', marker='o', markersize=4)
        if show_val:
            ax.plot(epochs, hist['val_acc'], label=f'{name} (Val)', marker='x', markersize=4, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
###
#5.
###

##############################################################
# accuracy function that returns the accuracy metrics
# As the classes are not balanced macro-F1 or balanced accuracy preferred over accuracy
# standard accuracy, balanced accuracy, macro-F1 score
##############################################################
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import torch

def print_evaluation_metrics(metrics_list):

    results = []

    for inst in metrics_list:
        
        
        model = inst['model']
        train_loader = inst['train_loader']
        val_loader = inst['val_loader']
        device = inst['device']
        model_name = inst['name']

        model.eval() # Set model to evaluation mode
        
        def get_predictions(loader):
            all_preds = []
            all_labels = []
            # no need for gradient calculation in inference
            with torch.no_grad(): 
                for inputs, labels in loader:
                    inputs = inputs.to(device)
                    
                    #fwd pass
                    outputs = model(inputs)
                    
                    #get the predicted class -> highest logit/prob
                    _, preds = torch.max(outputs, 1)
                    
                    #move back to CPU, convert to numpy
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    
            return all_labels, all_preds
    
        #get labels and predictions for both splits
        train_y, train_pred = get_predictions(train_loader)
        val_y, val_pred = get_predictions(val_loader)

        results.append({
            'name': model_name,
            'train_acc': accuracy_score(train_y, train_pred),
            'val_acc': accuracy_score(val_y, val_pred),
            'train_bal_acc': balanced_accuracy_score(train_y, train_pred),
            'val_bal_acc': balanced_accuracy_score(val_y, val_pred),
            'train_f1': f1_score(train_y, train_pred, average='macro'),
            'val_f1': f1_score(val_y, val_pred, average='macro')
        })
        

    header1 = f"{'':<20}"
    header2 = f"{'Metric':<20}"

    for res in results:
        header1 += f" | {res['name']:<23}"
        header2 += f" | {'Train':<10} | {'Validation':<10}"
        
    line_len = len(header1)
    
    print(f"\n{'-'*line_len}")
    print(header1)
    print(header2)
    print(f"{'-'*line_len}")

    row_acc = f"{'Standard Accuracy':<20}"
    row_bal = f"{'Balanced Accuracy':<20}"
    row_f1  = f"{'Macro F1 Score':<20}"
    
    for res in results:
        row_acc += f" | {res['train_acc']:<10.3f} | {res['val_acc']:<10.3f}"
        row_bal += f" | {res['train_bal_acc']:<10.3f} | {res['val_bal_acc']:<10.3f}"
        row_f1  += f" | {res['train_f1']:<10.3f} | {res['val_f1']:<10.3f}"

    print(row_acc)
    print(row_bal)
    print(row_f1)
    print(f"{'-'*line_len}\n")
    

In [ ]:
###
#7.2
#transform learning
###
# ImageNet normalisation (required for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

seq_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = GestureSequenceDataset(train_df, transform=seq_transform)
val_ds = GestureSequenceDataset(test_df, transform=seq_transform)

#batch_size=16 means 16 sequences * 5 frames = 80 images per batch
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

#resnet 18

#forward pass (Batch, Sequence, Channels, Height, Width)
def sequence_forward(self, x):
    
    B, seq_len, C, H, W = x.size()
    #flatten batch and sequence dimensions
    x = x.view(B * seq_len, C, H, W)
    #pass through forward method
    out = self.base_forward(x)
    #reshape back to sequences and average the predictions across the 5 frames
    out = out.view(B, seq_len, -1)
    out = out.mean(dim=1) 
    
    return out

    
#ResNet18 model, override forward method for sequences
def create_resnet18(num_classes):
    
    #load ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    #replace the final linear layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    #save the orig forward method
    model.base_forward = model.forward
    #bind specific impl sequence_forward function to this model instance
    model.forward = types.MethodType(sequence_forward, model)
    
    return model

#all layers except fully connected (fc)
def freeze_backbone(model):
    for name, param in model.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True

def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')

# Step 1: Linear probing
NUM_CLASSES = len(CLASS_NAMES)

torch.manual_seed(0)
baseline = create_resnet18(NUM_CLASSES).to(device)
freeze_backbone(baseline)

print('Step 1: Linear Probing')
count_params(baseline)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, baseline.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss()

all_results = {}
all_results['v1a: Linear Probing'] = run_experiment(
    baseline, optimizer, criterion, train_loader_v1, val_loader,
    epochs=5, name='v1a: Probing'
)




In [2]:
all_results = {}
all_results['v1: Linear Probing'] = run_experiment(
    model, optimizer, criterion, train_loader, val_loader,
    epochs=10, name='Sequence Probing'
)

NameError: name 'run_experiment' is not defined

In [ ]:
###
#6.
###

import torch
import torch.nn as nn
import torch.optim as optim

#########################
# 1. CNN_512_Grey
# 2. CNN_512_Grey_norm
#########################

#--------------------
#Base transforms
#--------------------

#CNN_512_Grey
base_transform_512_grey = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.Grayscale(num_output_channels=1),  # Convert to 1 channel
    transforms.ToTensor()
])

#CNN_512_Grey_with_normalization -> scales pixel values from a range of 0 to 1 down to a range of -1 to 1.
base_transform_512_grey_norm = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.Grayscale(num_output_channels=1),  # Convert to 1 channel
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])   # 1-channel normalization
])


#--------------------
#full datasets
#--------------------
#grey
full_train_ds_grey = GestureSequenceDataset(train_df, transform=base_transform_512_grey)
test_ds_grey = GestureSequenceDataset(test_df, transform=base_transform_512_grey)

#--------------------
#single split 0->677, 677-end
#--------------------
torch.manual_seed(42)
seq_indices = torch.randperm(len(full_train_ds_grey)).tolist()
train_indices = seq_indices[:677]
val_indices = seq_indices[677:]


train_ds_grey = Subset(full_train_ds_grey, train_indices)
val_ds_grey = Subset(full_train_ds_grey, val_indices)

#grey norm
full_train_ds_norm = GestureSequenceDataset(train_df, transform=base_transform_512_grey_norm)
test_ds_norm = GestureSequenceDataset(test_df, transform=base_transform_512_grey_norm)


train_ds_norm = Subset(full_train_ds_norm, train_indices)
val_ds_norm = Subset(full_train_ds_norm, val_indices)





#--------------------
#DataLoaders
#--------------------
#grey
train_loader_grey = DataLoader(
    train_ds_grey,
    batch_size=4,   # 4 sequences = 20 frames per batch
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_grey = DataLoader(
    val_ds_grey,
    batch_size=8,   # 8 sequences = 40 frames per batch
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

#grey norm
train_loader_norm = DataLoader(
    train_ds_norm,
    batch_size=4,   # 4 sequences = 20 frames per batch
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_norm = DataLoader(
    val_ds_norm,
    batch_size=8,   # 8 sequences = 40 frames per batch
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

#StratifiedKFold: item in full_train_ds_norm.instances is (uid, instance_df)
sequence_labels = [
    class_to_idx[inst_df.iloc[0]['gesture']]
    for _, inst_df in full_train_ds_norm.instances
]



In [ ]:
###
#8.
###

from sklearn.model_selection import StratifiedKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Training on device: {device}")

results = {}
epochs_grey_and_norm = 5 #number of epochs

num_classes = len(CLASS_NAMES)
criterion = nn.CrossEntropyLoss()

#--------------------
#Baseline Run: CNN_Grey
#--------------------
cnn_grey = CNN_512_Grey_EarlyFusion(num_classes).to(device)
optimizer_grey = optim.Adam(cnn_grey.parameters(), lr=1e-3)
results['CNN_Grey'] = run_experiment(
    cnn_grey, 
    optimizer_grey, 
    criterion, 
    train_loader_grey, 
    val_loader_grey,
    epochs=epochs_grey_and_norm, 
    name='CNN_Grey'
)

#--------------------
#Baseline Run: CNN_Grey_norm
#--------------------
cnn_grey_norm = CNN_512_Grey_EarlyFusion(num_classes).to(device)
optimizer_grey_norm = optim.Adam(cnn_grey_norm.parameters(), lr=1e-3)
results['CNN_Grey_norm'] = run_experiment(
    cnn_grey_norm, 
    optimizer_grey_norm, 
    criterion, 
    train_loader_norm, 
    val_loader_norm,
    epochs=epochs_grey_and_norm, 
    name='CNN_Grey_norm'
)

#--------------------
#Stratified K-Fold CNN_Grey_norm
#--------------------
num_splits = 5
skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

cv_models = []
cv_loaders = []
fold_results_list = [] #to hold each fold and then will be used to average

for fold, (train_idx, val_idx) in enumerate(skf.split(full_train_ds_norm, sequence_labels)):
    print(f"\nTarget fold {fold + 1}/{num_splits}")
    
    # Subsets for current fold
    train_ds_fold = Subset(full_train_ds_norm, train_idx)
    val_ds_fold   = Subset(full_train_ds_norm, val_idx)
    
    train_loader_fold = DataLoader(
        train_ds_fold, 
        batch_size=4, 
        shuffle=True, 
        num_workers=2, 
        pin_memory=True
    )
    val_loader_fold = DataLoader(
        val_ds_fold, 
        batch_size=8, 
        shuffle=False, 
        num_workers=2, 
        pin_memory=True
    )
    
    # Re-initialize model & optimizer per fold
    model_fold = CNN_512_Grey_EarlyFusion(num_classes).to(device)
    optimizer_fold = optim.Adam(model_fold.parameters(), lr=1e-3)
    
    # Run training for the fold
    fold_name = f'CNN_Grey_norm_Fold_{fold + 1}'
    fold_res = run_experiment(
        model_fold, 
        optimizer_fold, 
        criterion, 
        train_loader_fold, 
        val_loader_fold,
        epochs=epochs_grey_and_norm, 
        name=fold_name
    )
    fold_results_list.append(fold_res)
    
    # Store references for evaluation table
    cv_models.append(model_fold)
    cv_loaders.append((train_loader_fold, val_loader_fold))

#compute the average are add to the results for kfold
averaged_results = {}
if fold_results_list:
    metric_keys = fold_results_list[0].keys()

    for key in metric_keys:
        averaged_results[key] = []
        for epoch_idx in range(epochs_grey_and_norm):
            epoch_values_across_folds = [fold[key][epoch_idx] for fold in fold_results_list]

            mean_val = sum(epoch_values_across_folds) / num_splits
            averaged_results[key].append(mean_val)

results['CNN_Grey_norm_CV_Avg'] = averaged_results
    

plot_experiments(results)

In [ ]:
###
#7.
###

class CNN_512_Grey_EarlyFusion(nn.Module):
    def __init__(self, num_classes, num_frames=5):
        super().__init__()
        
        #in_channels = 5 (5 grayscale frames * 1 channel per frame)
        self.conv_block = nn.Sequential(
            # Input shape: (Batch, 5, 512, 512)
            nn.Conv2d(in_channels=num_frames, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((8, 8))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x shape from DataLoader: (Batch, 5, 1, 512, 512)
        B, T, C, H, W = x.shape
        
        # Merge Time (T) into Channels (C): shape becomes (Batch, 5, 512, 512)
        x = x.view(B, T * C, H, W)

        # Process sequence all at once in a single forward pass
        x = self.conv_block(x)
        x = self.adaptive_pool(x)
        x = self.classifier(x)
        # return (Batch, num_classes)
        return x 



In [ ]:
###
#9.
###

##################################################
# print scores on train and validation sets
##################################################
evaluation_metrics = []

evaluation_metrics.append({
    'name': 'Standard Greyscale',
    'model': cnn_grey,
    'train_loader': train_loader_grey,
    'val_loader': val_loader_grey,
    'device': device
})

evaluation_metrics.append({
    'name': 'Normalized Greyscale',
    'model': cnn_grey_norm,
    'train_loader': train_loader_norm,
    'val_loader': val_loader_norm,
    'device': device
})

# Stratified K-Fold evaluations
for fold_idx, (model_fold, (tr_loader, val_loader)) in enumerate(zip(cv_models, cv_loaders)):
    evaluation_metrics.append({
        'name': f'Norm_Grey (Fold {fold_idx + 1})',
        'model': model_fold,
        'train_loader': tr_loader,
        'val_loader': val_loader,
        'device': device
    })
    

print_evaluation_metrics(evaluation_metrics)
